# Planning with MCTS + `TCenterReward`: can the tree's edges reach *different* states?

**Problematic.** A Monte-Carlo tree search can only *search* (ie exploration) if the edges leaving a
node lead to **distinguishable** states: if every sibling edge collapses back onto
the same world-model trajectory, MCTS/UCB has nothing to choose between and the planning can't be performed. 

Experiments in this notebook are performed under the PushT framework.

1. **Part I: How to different edges? With what levers?**  
   - We investigate first to create diversity by injecting noise in the *context window* $s_{t-w:t}$ of the 'policy' $p(a_{t:t+H} | s_{t-w:t})$ (that we derived from the UWM formulation) ; or in the *context window*
   $s_{t-w:t}$ of the joint distribution $p(s_{t+1:t+1+H}, a_{t:t+H} | s_{t-w:t})$.  
   - We then investigate different metrics to quantify the diversity through the causal pipe 
   $action \rightarrow latent state \rightarrow task pose \rightarrow reward$:
      - the *entropy* (measure of dispersion) of the actions $p(a_{t:t+H} | s_{t-w:t})$
      - we investigate the "Vendi score" measured on the latent states of the rollouts - that is supposed to inform on the "effective number of distinguisable edges" - but this metric ends to not really be reliable...
      - the reward (computed at the last states of a batch of edges) standard deviation. This metric is the most useful since the reward score is the piece of information used in the MCTS algorithm to compute the UCB1 score.
   *Note:* We built a *simple reward function* based on the decoded latent states. On the decoded latent state, we localize the T-shaped object by its center. We then calculate the L1 distance between the T-shaped object and the center of the frame. The goal is to minimize this distance (reward implemented with a "gaussian kernel").

2. **Part II: Plans with `MCTS` + `TCenterReward`.**  
   The reward (red **T** *centered*) is deliberately simple - it is not
   the object of study; we use it to exercise planning with the world model. Goal:
   a plan that *raises* the reward from a bad starting configuration, and a sweep
   showing which planner parameters make planning beat random.

3. **Part III: Tree metrics.**  
We compute the tree statistics
   that actually diagnose exploration (visit-count entropy, root-child value
   spread, realized branching/depth, the visit↔value correlation, the terminal-
   reward spread *inside* the tree).

Everything runs in tokenizer-latent space under **bf16** (fp32 OOMs at planning
batch sizes) on the local GPU (`conda env dreamerv4uwm`).

## Setup: model, data, helpers

In [ ]:
%load_ext autoreload
%autoreload 2
import math, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import mediapy
from torch.nn.functional import interpolate
import cv2

torch.manual_seed(0)                       # reproducible noise priors
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
resolution = (256, 256)
print('device:', device)

In [ ]:
# --- config + checkpoints ---------------------------------------------------
# This notebook lives in dreamerv4uwm/planning/, so the diverse-tests relative
# `../scripts/config` no longer resolves. Anchor the hydra config dir to the
# installed package instead (robust to the kernel's working directory).
import dreamerv4uwm
from hydra import initialize_config_dir, compose

CONFIG_DIR = str(Path(dreamerv4uwm.__file__).resolve().parent.parent / 'scripts' / 'config')
with initialize_config_dir(version_base=None, config_dir=CONFIG_DIR):
    cfg = compose(config_name='dynamics/pushT-large',
                  overrides=['denoiser.horizon_aware=false'])

cfg.dynamics_ckpt  = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/blockcausal/pushT-post-train/97500.pt'
cfg.tokenizer_ckpt = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer/pushT.pt'
print('config:', CONFIG_DIR)
print('horizon_aware:', cfg.denoiser.get('horizon_aware', False),
      '| n_actions:', cfg.denoiser.n_actions,
      '| latent grid:', cfg.denoiser.num_latent_tokens, 'x', cfg.denoiser.latent_dim)

In [ ]:
from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser

denoiser  = load_denoiser(cfg, device, max_num_forward_steps=300).eval().cuda()
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300).eval().cuda()
print('models loaded  |  frame_id_embedder:', denoiser.model.frame_id_embedder)

In [ ]:
# --- display / decode / scoring helpers ------------------------------------
@torch.no_grad()
def decode(lat):
    """Latents (B,T,N,D) -> video (B,T,3,H,W) float[0,1] (bf16 autocast)."""
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        v = tokenizer.decode(lat.to(device)) #shape (B,T,3,H,W)
    return v.float().clamp(0, 1)

@torch.no_grad()
def rgb_batch(z_states):
    """Terminal latents (B,N,D) -> (B,H,W,3) uint8 RGB (one batched decode)."""
    v = decode(z_states[:, None])[:, 0]                       # (B,3,H,W)
    return (v.permute(0, 2, 3, 1).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy() # (B,H,W,3) uint8

def frame_rgb(lat_1frame):
    """(1,1,N,D) latent -> (H,W,3) uint8 RGB."""
    v = decode(lat_1frame)[0, 0] # (3,H,W)
    return (v.permute(1, 2, 0).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy() # (H,W,3) uint8

def _strip(video_tchw, n_frames=8, border=2):
    """ Stack frames of a (T,C,H,W) video as a single (H,W,C) filmstrip with white borders.
    Args:
        video_tchw: (T,C,H,W) torch tensor
        n_frames: number of frames to include in the strip (evenly spaced)
        border: width of white border between frames
    Returns:
        (H,W,C) uint8 numpy array
    """
    T = video_tchw.shape[0]
    idx = np.linspace(0, T - 1, min(n_frames, T), dtype=int)
    # Convert to (T,H,W,C) uint8 numpy array
    fr = (video_tchw[idx].cpu().permute(0, 2, 3, 1).float().numpy() * 255).clip(0, 255).astype(np.uint8)
    H, W, C = fr.shape[1:]
    sep = np.full((H, border, C), 255, np.uint8)
    parts = []
    for i, f in enumerate(fr):
        if i: 
            parts.append(sep)
        parts.append(f)
    return np.concatenate(parts, axis=1) # (H, W*n_frames + border*(n_frames-1), C)

def plotComparison(named_videos, n_frames=8, title=None):
    """Stack labeled (T,C,H,W) clips as filmstrip rows (ie. one row per clip, one column per frame).
    Args:
        named_videos: list of (label, video) pairs, where video is (T,C,H,W) torch tensor
        n_frames: number of frames to include in the strip (evenly spaced)
        title: optional title for the figure
    Returns:
        None (displays a matplotlib figure)
    """
    rows = [(lab, _strip(v, n_frames)) for lab, v in named_videos]
    fig, axes = plt.subplots(len(rows), 1, figsize=(n_frames * 2, 2.0 * len(rows)))
    if len(rows) == 1: axes = [axes]
    for ax, (lab, img) in zip(axes, rows):
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(lab, rotation=0, ha='right', va='center', fontsize=10)
    if title: axes[0].set_title(title, fontsize=12)
    plt.tight_layout(); plt.show()

def plotVideo(video, fps=10):
    """Play a (T, C, H, W) float[0,1] clip inline.
    Display the frames' number for each frame."""
    arr = (video.cpu().permute(0, 2, 3, 1).to(torch.float32).numpy() * 255).clip(0, 255).astype(np.uint8)
    for i in range(arr.shape[0]):
        arr[i] = np.pad(arr[i], ((0, 0), (0, 0), (0, 0)), mode='constant', constant_values=255)
        cv2.putText(arr[i], f'Frame {i}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    mediapy.show_video(arr, fps=fps)

def get_window(idx):
    """Get a window of (image, action) from the dataset."""
    batch = dataset[idx]
    imgs = interpolate(batch['image'], resolution).to(device)[None]           # (1,T,3,256,256)
    actions = batch['action'][:, :cfg.denoiser.n_actions][None].to(device)   # (1,T,n_act)
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs).float()                             # (1,T,N_lat,D_lat)
    return imgs, actions, latents

In [ ]:
# --- a held-out window -> latents (rich T motion; used for Part I) ----------
from dreamerv4uwm.datasets import ShardedHDF5Dataset

DATA_PATH = '/home/mim-server/datasets/pushT/h5/play'
# seed and shuffle_windows=False to get a reproducible window (otherwise the window is randomly sampled from the dataset)
dataset = ShardedHDF5Dataset(data_dir=DATA_PATH, window_size=64, stride=1,
                             split='train', train_fraction=0.9, split_seed=123, shuffle_windows=False)

# size of the dataset (number of windows)
print('dataset size:', len(dataset))

# fix a random seed to get a reproducible window
WINDOW_SEED = 1111
# Get a window of (image, action) from the dataset.
imgs, actions, latents = get_window(WINDOW_SEED)

print('imgs', tuple(imgs.shape), '| actions', tuple(actions.shape), '| latents', tuple(latents.shape))
plotVideo(imgs[0], fps=10)

In [ ]:
# --- context / horizon split ------------
T_CTX = 8                                     # conditioning frames a node holds
N_SAMPLES = 48             # edges / action samples per estimate
K_STEPS   = 8              # Euler steps per rollout

def make_state(t0, Tc=T_CTX, window_seed=WINDOW_SEED, visualize=False):
    """(ctx_z, ctx_a, gt_next_action) for a decision at frame t0+Tc of the demo window."""
    _, actions, latents = get_window(window_seed)
    cz = latents[:, t0:t0 + Tc].clone()       # shape (1,Tc,N_lat,D_lat) dataset latents
    ca = actions[:, t0:t0 + Tc].clone()       # shape (1,Tc,n_act) dataset actions
    gt = actions[:, t0 + Tc].clone()          # (1,n_act) dataset action
    if visualize:
        plotComparison([('context', decode(cz)[0])], n_frames=Tc + 1)
    return cz, ca, gt

print(f'T_ctx={T_CTX}  N_SAMPLES={N_SAMPLES}  K_STEPS={K_STEPS}')

In [ ]:
# ------ Visualize one entire rollout, showing the decoded frames and the reward scores for each frame in the rollout.
def visualize_rollout(cz, ca, H=8, B=1, ctx_noise=0.3, action_temp=1.0, K=K_STEPS, seed=42):
    """Visualize a single rollout from the given context and actions."""
    if seed is not None:
        g = torch.Generator(device=device).manual_seed(seed)
    else:
        g = None
    z, a = R.imagine(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                     action_temp=action_temp, generator=g)
    
    z_rgb = rgb_batch(z[0])  # (H,W,3)
    scores = []
    for h in range(H):
        score, _ = score_t_centered(z_rgb[h], **SCORE_KW)
        scores.append(score)

    # Display the rollout frames with their corresponding reward scores
    # 8 visualizations per row, with the reward score in the title
    n_rows = math.ceil(H / 8)
    fig, axes = plt.subplots(n_rows, 8, figsize=(16, 2 * n_rows))
    for h in range(H):
        r, c = divmod(h, 8)
        ax = axes[r, c] if n_rows > 1 else axes[c]
        ax.imshow(z_rgb[h])
        ax.set_title(f'Frame {h}\nReward: {scores[h]:.2f}')
        ax.axis('off')
    fig.suptitle(f'Rollout visualization: ctx_noise={ctx_noise:.2f}, action_temp={action_temp:.2f}')
    plt.tight_layout()
    plt.show()

## Diversity metrics

The core question of the whole notebook: *do sibling edges reach different
states?* — is a **diversity** question, asked at four levels:

| level | what it is | why it matters for MCTS |
|---|---|---|
| **action** `a` | the control the edge applies | the raw knob we can perturb |
| **latent state** `z` | the world-model state the child node *holds* | what the tree branches into |
| **task pose** | decoded T centroid `(x,y)` | the task-relevant part of `z` (latent diversity can be task-*irrelevant*) |
| **reward** `r` | the scalar the planner selects on | if it is flat, the tree cannot decide |


Beyond spread/entropy we compute the **effective number of distinguishable
samples** with a *Vendi score* — the exponentiated entropy of the eigenvalues of
an RBF similarity matrix (bandwidth = median pairwise distance). It reads as "how
many effectively-distinct edges are here": `1` = all collapsed, `B` = all distinct.
This is the single most on-target statistic for the problematic.  
*Note: this metric actually doesn't work very well when computed on the latent state. More on that at the end of the notebook*

In [ ]:
# --- diversity primitives ---------------------------------------------------
def _to2d(X):
    if torch.is_tensor(X): 
        X = X.detach().float().cpu().numpy()
    else:                  
        X = np.asarray(X, dtype=np.float64)
    return X.reshape(X.shape[0], -1).astype(np.float64)

def pairwise_dist(X):
    """(B,...) -> (B,B) euclidean distances over the flattened features."""
    Xt = torch.from_numpy(_to2d(X)).float()
    return torch.cdist(Xt, Xt).double().numpy()

def mean_pairwise_dist(X):
    D = pairwise_dist(X); B = D.shape[0]
    return float(D.sum() / (B * (B - 1) + 1e-9))

def vendi_score(X, sigma=None):
    """Effective number of distinguishable samples in [1, B] (RBF-kernel Vendi)."""
    Xf = _to2d(X); B = Xf.shape[0]
    if B < 2: return 1.0
    D = pairwise_dist(Xf); off = D[~np.eye(B, dtype=bool)]
    if sigma is None:
        pos = off[off > 0]; sigma = float(np.median(pos)) if pos.size else 1.0
    if sigma <= 0: return 1.0
    K = np.exp(-(D ** 2) / (2 * sigma ** 2))
    w = np.linalg.eigvalsh(K / B); w = w[w > 1e-12]
    return float(np.exp(-(w * np.log(w)).sum()))

def participation_ratio(X):
    """Effective dimensionality of the sample cloud (covariance eigenvalues)."""
    Xf = _to2d(X); Xc = Xf - Xf.mean(0, keepdims=True)
    lam = np.linalg.svd(Xc, compute_uv=False) ** 2
    return float(lam.sum() ** 2 / (lam ** 2).sum()) if lam.sum() > 0 else 0.0

def gaussian_entropy(samples):
    """Differential entropy (nats) of a Gaussian fit to samples (B,d) — small d only."""
    x = _to2d(samples); d = x.shape[1]
    cov = np.cov(x.T).reshape(d, d) + 1e-9 * np.eye(d)
    # (1/2) * (d * log(2*pi*e) + log(det(cov))) = (1/2) * (d * log(2*pi*e) + slogdet(cov)[1])
    return float(0.5 * (d * math.log(2 * math.pi * math.e) + np.linalg.slogdet(cov)[1]))

In [ ]:
# --- task/reward diversity metrics ---------------------------------------------------
from dreamerv4uwm.planning import (rollout as R, MCTS, PlanConfig,
                                   TCenterReward, score_t_centered, annotate_t)
from dreamerv4uwm.planning.reward import score_loc

# score kwargs shared by the reward and the task-pose readout (kept identical so
# the reward we report IS TCenterReward's value, computed from the same decode).
SCORE_KW = dict(center_xy=(0.5, 0.5), sigma=0.25)

def task_poses(rgb_np, **kw):
    """(B,H,W,3) uint8 -> (poses (m,2): [cx_norm, cy_norm], scores (B,)).
    cx_norm, cy_norm are the normalized centroid coordinates in [0,1]."""
    kw = {**SCORE_KW, **kw}
    scores, poses = [], []
    Himg, Wimg = rgb_np.shape[1:3]
    for b in range(rgb_np.shape[0]):
        s, d = score_t_centered(rgb_np[b], **kw)
        scores.append(s)
        if d.get('found'):
            cx, cy = d['centroid']
            poses.append([cx / Wimg, cy / Himg])
    return np.array(poses), np.array(scores)

def reward_std(inputs, inputs_lat=True):
    """Compute the standard deviation of the reward across a batch of inputs.
    Args:
        inputs: (B,T,C,H,W) or (B,T,N,D) tensor of inputs (images or latents)
        inputs_lat: whether the inputs are latents (True) or images (False)
    Returns:
        std: standard deviation of the reward across the batch
    """
    if inputs_lat:
        rgb = rgb_batch(inputs)  # decode latents to RGB
    else:
        rgb = (inputs.permute(0, 2, 3, 4, 1).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)  # images to RGB
    _, scores = task_poses(rgb)
    return float(scores.std())

def poses_std(inputs, inputs_lat=True):
    """Compute the standard deviation of the poses across a batch of inputs.
    Args:
        inputs: (B,T,C,H,W) or (B,T,N,D) tensor of inputs (images or latents)
        inputs_lat: whether the inputs are latents (True) or images (False)
    Returns:
        std: standard deviation of the poses across the batch
    """
    if inputs_lat:
        rgb = rgb_batch(inputs)  # decode latents to RGB
    else:
        rgb = (inputs.permute(0, 2, 3, 4, 1).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)  # images to RGB
    poses, _ = task_poses(rgb)
    return np.std(poses, axis=0) if poses.size > 0 else 0.0

In [ ]:
# Check that the diversity metrics are consistent with each other on a small random sample.
def compare_diversity_metrics(frame_indices, window_seed=WINDOW_SEED):
    """Compute and compare diversity metrics for selected frames."""
    imgs, _, latents = get_window(window_seed)

    latents_of_interest = latents[:, frame_indices]  # (1, len(frame_indices), N, D)
    imgs_of_interest = imgs[:, frame_indices]  # (1, len(frame_indices), 3, H, W)
    selected_frames_rgb = rgb_batch(latents_of_interest[0])  # (len(frame_indices), H, W, 3)

    vendi = vendi_score(latents_of_interest[0])
    mean_dist = mean_pairwise_dist(latents_of_interest[0])
    part_ratio = participation_ratio(latents_of_interest[0])
    gauss_entropy = gaussian_entropy(latents_of_interest[0])
    reward_std_dev = reward_std(latents_of_interest[0], inputs_lat=True)
    # poses_std_dev = poses_std(latents_of_interest[0], inputs_lat=True)

    # Visualize the selected frames (imgs and decoded latents)
    fig, axes = plt.subplots(2, len(frame_indices), figsize=(16, 4))
    for i, ax in enumerate(axes[0]):
        idx = frame_indices[i]
        ax.imshow(imgs_of_interest[0, i].permute(1, 2, 0).cpu().numpy())
        ax.set_title(f'Original Frame {idx}')
        ax.axis('off')
    for i, ax in enumerate(axes[1]):
        ax.imshow(selected_frames_rgb[i])
        ax.set_title(f'Decoded Frame {frame_indices[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Vendi score: {vendi:.2f}')
    print(f'Mean pairwise distance: {mean_dist:.4f}')
    print(f'Participation ratio: {part_ratio:.2f}')
    print(f'Gaussian entropy: {gauss_entropy:.4f}')
    print(f'Reward standard deviation: {reward_std_dev:.4f}')
    # print(f'Poses standard deviation: {poses_std_dev}')

In [ ]:
compare_diversity_metrics(frame_indices=[0, 8, 16, 24, 32, 40, 48, 56])
compare_diversity_metrics(frame_indices=[0, 1, 2, 3, 4, 5, 6, 7])
compare_diversity_metrics(frame_indices=[0, 0, 0, 0, 0, 0, 0, 0])
compare_diversity_metrics(frame_indices=[0, 0, 0, 0, 0, 0, 0, 10])

## Quick study of the reward function

In [ ]:
from dreamerv4uwm.planning.reward import score_loc

# Display a heatmap of the reward scores ON one frame of the window to visualize how the reward varies spatially. This can help in understanding the areas of interest for the task.
def display_reward_heatmap(frame_index, window_seed=WINDOW_SEED, center_xy=(0.5, 0.5), sigma_values=[0.1, 0.25, 0.5]):
    """Display a heatmap of the reward scores on a specific frame in the demo window for different sigma values."""
    imgs, _, _ = get_window(window_seed)
    img = imgs[0, frame_index]  # (3, H, W) float[0,1]
    img_rgb = (img.permute(1, 2, 0).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()  # (H,W,3) uint8

    H, W = img_rgb.shape[:2]
    x_coords = np.linspace(0, W - 1, W)
    y_coords = np.linspace(0, H - 1, H)
    xv, yv = np.meshgrid(x_coords, y_coords)

    fig, axes = plt.subplots(1, len(sigma_values), figsize=(16, 4))
    for i, sigma in enumerate(sigma_values):
        heatmap = np.zeros((H, W), dtype=np.float32)
        for y in range(H):
            for x in range(W):
                loc_xy = (x / W, y / H)  # normalized coordinates
                heatmap[y, x] = score_loc(loc_xy, center_xy=center_xy, sigma=sigma)

        axes[i].imshow(img_rgb)
        im = axes[i].imshow(heatmap, cmap='jet', alpha=0.5)
        fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
        axes[i].set_title(f'Frame {frame_index} | Sigma: {sigma}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

display_reward_heatmap(frame_index=32, sigma_values=[0.1, 0.25, 0.3, 0.4, 0.5])

In [ ]:
# Display the reward score on a few frames of the window to verify that the scoring function is working as expected.
def display_reward_scores(frame_indices, window_seed=WINDOW_SEED, **kw):
    """Display the reward scores for selected frames in the demo window."""
    kw = {**SCORE_KW, **kw}
    imgs, _, _ = get_window(window_seed)
    selected_imgs = imgs[:, frame_indices]  # (1, n, 3, H, W) float[0,1]
    scores = []
    for b in range(selected_imgs.shape[1]):
        # score_t_centered wants (H,W,3) uint8 [0,255], NOT float [0,1]
        processed_img = (selected_imgs[0, b].permute(1, 2, 0).clamp(0, 1) * 255
                         ).to(torch.uint8).cpu().numpy()          # (H,W,3) uint8
        score, _ = score_t_centered(processed_img, **kw)
        scores.append(score)

    fig, axes = plt.subplots(1, len(frame_indices), figsize=(16, 4))
    for i, ax in enumerate(axes):
        ax.imshow(selected_imgs[0, i].permute(1, 2, 0).cpu().numpy())   # imshow is fine with float[0,1]
        ax.set_title(f'Frame {frame_indices[i]}\nReward: {scores[i]:.2f}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
display_reward_scores(frame_indices=[0, 8, 16, 24, 32, 40, 48, 56])

## Metrics on edges

In [ ]:
def characterize_edges(cz, ca, *, H, B, ctx_noise=0.0, action_temp=1.0,
                       K=None, mode='imagine', seed=0, task=True, visualize=False):
    """Sample B edges from a node and measure diversity at every funnel level.
    Returns (metrics_dict, (z (B,H,N,D), a (B,H,n_act)))."""

    K = K or K_STEPS
    g = torch.Generator(device=device).manual_seed(seed)

    if mode == 'two_stage':                          # p(a|o) then world-model s,a->s'
        a = R.policy(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                     action_temp=action_temp, generator=g)
        z = R.transition(denoiser, cz, ca, a, K=K, ctx_noise=ctx_noise, generator=g)
    else:                                            # joint imagine  p(o',a|o)
        z, a = R.imagine(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                         action_temp=action_temp, generator=g)
        
    z_term = z[:, -1]                                # (B,N,D) child states
    m = dict(H=H, B=B, ctx_noise=ctx_noise, action_temp=action_temp, mode=mode)
    a0 = a[:, 0]                                     # first action of each edge

    # m['act_mpd'] = mean_pairwise_dist(a0)
    # m['act_vendi'] = vendi_score(a0)
    m['act_entropy'] = gaussian_entropy(a0)
    m['act_spread'] = float(a.std(0).norm(dim=-1).mean())
    m['lat_spread'] = float(z_term.std(0).mean())
    m['lat_mpd'] = mean_pairwise_dist(z_term)
    m['lat_vendi'] = vendi_score(z_term)
    m['lat_pr'] = participation_ratio(z_term)
    if task:
        P, r = task_poses(rgb_batch(z_term))
        m['rew_mean'] = float(r.mean())
        m['rew_std'] = float(r.std())
        m['rew_range'] = float(r.max() - r.min())
        m['task_found'] = len(P)
        if len(P) >= 2:
            m['task_centroid_spread'] = float(np.linalg.norm(P.std(0)))
            m['task_vendi'] = vendi_score(P)
        else:
            m['task_centroid_spread'] = 0.0; m['task_vendi'] = 1.0
    
    # if visualize: display the decoded frames of the child states
    # 8 visualizations per row, with the reward score in the title
    # Main title: "Edge characterization: ctx_noise=..., action_temp=..., mode=..."
    if visualize:
        z_rgb = rgb_batch(z_term)  # (B,H,W,3)
        n_rows = math.ceil(B / 8)
        fig, axes = plt.subplots(n_rows, 8, figsize=(16, 2 * n_rows))
        for i in range(B):
            r, c = divmod(i, 8)
            ax = axes[r, c] if n_rows > 1 else axes[c]
            ax.imshow(z_rgb[i])
            rwd = score_t_centered(z_rgb[i], **SCORE_KW)[0]
            ax.set_title(f'Edge {i}\nReward: {rwd:.2f}')
            ax.axis('off')
        fig.suptitle(f'Edge characterization: H={H}, ctx_noise={ctx_noise:.2f}, action_temp={action_temp:.2f}, mode={mode}')
        plt.tight_layout()
        plt.show()

    return m, (z, a)

# Part I: How to make different edges?

The world model is the binding constraint: the policy `p(a|o)` is often a tight
blob, and even when we *widen* the action distribution the model tends to
**contract** those diverse actions back toward one short-horizon trajectory. So
"how do we generate edges that lead to different states?" is really: *which knobs
survive the funnel down to distinguishable states / rewards, and at what horizon?*

## 1. Action diversity $p(a_t | o_{t-w:t})$ - context-noise injection

In [ ]:
# Get ground truth action statistics over the entire window
gt_actions = get_window(WINDOW_SEED)[1][0]  # (T, n_act)
gt_action_mean = gt_actions.mean(0)
gt_action_std = gt_actions.std(0)
gt_action_max = gt_actions.max(0).values
gt_action_min = gt_actions.min(0).values

# Display the ground truth action statistics on a box plot
plt.figure(figsize=(10, 6))
plt.boxplot(gt_actions.cpu().numpy(), labels=[f'Action {i}' for i in range(gt_actions.shape[1])])
plt.xlabel('Actions')
plt.ylabel('Values')
plt.title('Ground Truth Action Statistics')
plt.show()

# Print the ground truth action statistics
print("Ground Truth Action Mean:", gt_action_mean.cpu().numpy())
print("Ground Truth Action Std Dev:", gt_action_std.cpu().numpy())
print("Ground Truth Action Max:", gt_action_max.cpu().numpy())
print("Ground Truth Action Min:", gt_action_min.cpu().numpy())

In [ ]:
t0 = 24
cz, ca, gt = make_state(t0)
NOISES = [0.0, 0.15, 0.3, 0.5, 0.7]

clouds, ent_h, ent_m, vend_h = {}, [], [], []
for n in NOISES:
    g = torch.Generator(device=device).manual_seed(0)
    a_h = R.policy(denoiser, cz, ca, H=1, B=N_SAMPLES, K=K_STEPS,
                   ctx_noise=n, ctx_noise_honest=True,  generator=g)[:, 0]
    clouds[f'n={n}'] = a_h.cpu().numpy()
    ent_h.append(gaussian_entropy(a_h))
    print(f'ctx_noise={n:.2f}  H={ent_h[-1]:+.2f}')


fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
for lab, c in clouds.items():
    ax[0].scatter(c[:, 0], c[:, 1], s=12, alpha=0.45, label=lab)
ax[0].scatter([gt[0, 0].item()], [gt[0, 1].item()], c='k', marker='*', s=220, label='dataset $a_t$', zorder=5)
ax[0].set_xlabel('action[0]'); ax[0].set_ylabel('action[1]'); ax[0].legend(fontsize=8)
ax[0].grid(alpha=.3); ax[0].set_title(f'ctx-noise action clouds')
ax[1].plot(NOISES, ent_h, 'o-', label='honest')
ax[1].set_xlabel('ctx_noise'); ax[1].set_ylabel('action entropy (nats)')
# ax[1].legend()
ax[1].grid(alpha=.3)
ax[1].set_title('entropy vs context noise')
plt.tight_layout(); plt.show()

## 2. Edge diversity - does context-noise injection leads to a diversity of task poses / rewards?

In [ ]:
WINDOW_SEED=2400     # t0=0, WINDOW_SEED= 3000
imgs, actions, latents = get_window(WINDOW_SEED)
plotVideo(imgs[0], fps=10)

cz, ca, gt = make_state(t0=0, Tc=T_CTX, window_seed=WINDOW_SEED, visualize=True)

In [ ]:
# Visualize one entire rollout with ctx_noise=0.3 and action_temp=1.0, showing the decoded frames and the reward scores for each frame in the rollout.
visualize_rollout(cz, ca, H=32, B=1, ctx_noise=0.3, action_temp=1.0, K=K_STEPS, seed=None)

In [ ]:
# Create a visualize rollout but without title, just the frames
def visualize_rollout_v2(cz, ca, H=8, B=1, ctx_noise=0.3, action_temp=1.0,
                         K=K_STEPS, seed=42, per_row=8):
    """Like visualize_rollout, but plain: just the decoded frames (no titles)."""
    g = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
    z, _ = R.imagine(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                     action_temp=action_temp, generator=g)

    z_rgb = rgb_batch(z[0])                                  # (H, H_img, W_img, 3)
    n_rows = math.ceil(H / per_row)
    fig, axes = plt.subplots(n_rows, per_row, figsize=(2 * per_row, 2 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for h in range(H):
        axes[h].imshow(z_rgb[h])
    for ax in axes:                    # blank every cell, incl. unused trailing ones
        ax.axis('off')
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    plt.show()


In [ ]:
visualize_rollout_v2(cz, ca, H=32, B=1, ctx_noise=0.3, action_temp=1.0, K=K_STEPS, seed=None)

In [ ]:
H_EDGE = 32
funnel = [characterize_edges(cz, ca, H=H_EDGE, B=N_SAMPLES, ctx_noise=n, mode='imagine',
                             action_temp=1.0, seed=41, visualize=False)[0] for n in NOISES]

In [ ]:
for m in funnel:
    print(f"ctx_noise={m['ctx_noise']:.2f}  " #act_vendi={m['act_vendi']:5.1f}  
          f"lat_vendi={m['lat_vendi']:4.1f}  task_vendi={m['task_vendi']:4.1f}  "
          f"rew_std={m['rew_std']:.3f}  task_centroid_spread={m['task_centroid_spread']:.3f}")

def _norm(key):
    v = np.array([m[key] for m in funnel]); 
    return v / (v[0] + 1e-9)

fig, ax = plt.subplots(1, 1, figsize=(12, 4.3)) # plt.subplots(1, 2, figsize=(12, 4.3))

lst = [
    # ('lat_vendi', 'latent state', 's-'),
    # ('act_vendi', 'action', 'o-'),
    # ('task_vendi', 'task pose', '^-'), 
    ('rew_std', 'reward std', 'd-')
    ]

for key, lab, mk in lst: 
    # ax[0].plot(NOISES, _norm(key), mk, label=lab)
    ax.plot(NOISES, np.array([m[key] for m in funnel]), mk, label=lab)

ax.set_xlabel('ctx_noise'); ax.set_ylabel('reward std')
ax.legend(); ax.grid(alpha=.3); ax.set_title(f'reward std (on B={N_SAMPLES} rollouts) vs ctx_noise \n(length of edges (horizon): H={H_EDGE})')

# ax[1].plot(NOISES, [m['act_vendi'] for m in funnel], 'o-', label='action')
# ax[1].plot(NOISES, [m['lat_vendi'] for m in funnel], 's-', label='latent state')
# ax[1].plot(NOISES, [m['task_vendi'] for m in funnel], '^-', label='task pose')
# ax[1].set_xlabel('ctx_noise'); ax[1].set_ylabel('effective # distinct edges (Vendi)')
# ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('absolute distinguishable edges per level')

plt.tight_layout(); plt.show()

## 3. What horizon and ctx_noise to choose?

In [ ]:
GRID_NOISE = [0.0, 0.3, 0.5, 0.7, 1.0]
GRID_H     = [4, 8, 12, 16, 20, 24, 28, 32]
lat_grid = np.zeros((len(GRID_NOISE), len(GRID_H)))
rew_grid = np.zeros_like(lat_grid)
Bg = 32 #12 if FAST else 20
for i, n in enumerate(GRID_NOISE):
    for j, H in enumerate(GRID_H):
        m, _ = characterize_edges(cz, ca, H=H, B=Bg, ctx_noise=n, action_temp=1.0, seed=3, visualize=True)
        lat_grid[i, j] = m['lat_vendi']
        rew_grid[i, j] = m['rew_std']
        print(f'ctx_noise={n:.2f}  H={H}  lat_vendi={m["lat_vendi"]:.1f}  rew_std={m["rew_std"]:.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for a_, G, ttl in [(ax[0], lat_grid, f'vendi score\neffective # distinct terminal states (ouf of {Bg})'),
                   (ax[1], rew_grid, 'terminal-reward std (what UCB1 sees)')]:
    im = a_.imshow(G, origin='lower', aspect='auto', cmap='viridis')
    a_.set_xticks(range(len(GRID_H)));    a_.set_xticklabels(GRID_H)
    a_.set_yticks(range(len(GRID_NOISE))); a_.set_yticklabels(GRID_NOISE)
    a_.set_xlabel('edge horizon H'); a_.set_ylabel('ctx_noise'); a_.set_title(ttl)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            a_.text(j, i, f'{G[i, j]:.2f}', ha='center', va='center',
                    color='w' if G[i, j] < G.max() * 0.6 else 'k', fontsize=9)
    fig.colorbar(im, ax=a_, fraction=0.046)
plt.suptitle('Edge-generation: how to make the edges starting from the same context diverse?', y=1.02)
plt.tight_layout(); plt.show()

## 4. Edge mode — joint `imagine` $p(o',a|o)$ vs `two-stage` $p(a|o) → s,a→s'$

`MCTS` can generate edges two ways. Joint imagination samples action and state
*together*; two-stage samples an action from the policy then rolls the world model
under it. Which gives more distinguishable children?

In [ ]:
modes = ['imagine', 'two_stage']
mrows = {}
H_e = 32
ctx_noise_e = 0.7
for md_ in modes:
    m, _ = characterize_edges(cz, ca, H=H_e, B=N_SAMPLES, ctx_noise=ctx_noise_e, action_temp=1.0,
                              mode=md_, seed=4)
    mrows[md_] = m
    print(f"{md_:10s}  lat_vendi={m['lat_vendi']:4.1f}  " #   act_vendi={m['act_vendi']:5.1f}
          f"task_vendi={m['task_vendi']:4.1f}  rew_std={m['rew_std']:.3f}")

labels = ['lat_vendi', 'task_vendi'] #'act_vendi', 
x = np.arange(len(labels)); w = 0.35
# fig, ax = plt.subplots(figsize=(7.5, 4))
# for k, md_ in enumerate(modes):
#     ax.bar(x + (k - 0.5) * w, [mrows[md_][l] for l in labels], w, label=md_)
# ax.set_xticks(x); ax.set_xticklabels(['latent', 'task pose']) #'action', 
# ax.set_ylabel('effective # distinct edges (Vendi)'); ax.legend()
# ax.set_title('joint imagine vs two-stage edges')
# plt.tight_layout(); plt.show()

In [ ]:
labels = ['rew_std'] #'act_vendi', 
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4))
for k, md_ in enumerate(modes):
    ax.bar(x + (k - 0.5) * w, [mrows[md_][l] for l in labels], w, label=md_)
ax.set_xticks(x) #ax.set_xticklabels(['reward std']) #'action', 
ax.set_ylabel('reward std')
ax.legend()
ax.set_title('joint imagine vs two-stage edges')
plt.tight_layout()
plt.show()

# Part II: Plans with `MCTS` + `TCenterReward`

The reward is a simple pixel-space task score — the red **T** *centered*
— used only to exercise planning; it is **not** the object of study. Goal: start
from a bad configuration and find a plan that **raises** the reward, then see which
planner parameters make planning beat random.

In [ ]:
SCORE_KW = dict(center_xy=(0.5, 0.5), sigma=0.25)
reward_T = TCenterReward(decode_fn=decode, **SCORE_KW)

WINDOW_SEED=1234    # t0=0, WINDOW_SEED= 3000
t0 = 50
imgs, actions, latents = get_window(WINDOW_SEED)
plotVideo(imgs[0], fps=10)

# cz shapes: (1, T_ctx, N, D), ca shapes: (1, T_ctx, n_act)
cz, ca, gt = make_state(t0=t0, Tc=T_CTX, window_seed=WINDOW_SEED, visualize=True)


s0, d0 = score_t_centered(frame_rgb(cz[:, -1:]), **SCORE_KW)
plt.figure(figsize=(4, 4))
plt.imshow(annotate_t(frame_rgb(cz[:, -1:]), d0))
plt.axis('off')
plt.title(f'last context frame\nR={s0:.2f}  center={d0["center"]:.2f}')
plt.tight_layout()
plt.show()
print(f'start reward = {s0:.3f}')

## 1. One plan, end to end

In [ ]:
epcfg = PlanConfig(horizon=28, branching=5, sim_horizon=28, sim_rollouts=3,
                       n_iterations=20, max_depth=3, c_ucb=0.5,
                       gamma=0.98, n_min=0, K_steps=6, ctx_noise=0.7,
                       action_temp=1.0, max_ctx=16)

planner0 = MCTS(denoiser, reward_T, epcfg, seed=0)
t = time.time()
out0 = planner0.plan(cz, ca, verbose=True)
plan_z = torch.cat([e.z_seq for e in out0['best_path']], 0)[None]
print(f'planned in {time.time()-t:.1f}s | forwards={out0["n_forward"]} | nodes={len(planner0.all_nodes)}')


In [ ]:
def peak_along(z_states):
    """best TCenterReward over the frames of a rollout (MPC-style, matches Eq.5 max-over-prefix)."""
    rgb = rgb_batch(z_states)
    _, sc = task_poses(rgb)
    return float(sc.max()), sc

def random_best_rollout(cfg_like, cz, ca, B=12, seed=7, extra=None):
    """B random `imagine` rollouts from `cz` (NO planning), same knobs as the plan's edges.
    Returns (rand_z (B, L, N, D) full trajectories, peak (B,) best reward per rollout)."""
    p = extra or {}
    g = torch.Generator(device=device).manual_seed(seed)

    zc = cz.expand(B, -1, -1, -1).contiguous()
    ac = ca.expand(B, -1, -1).contiguous()

    chunks = []
    peak = torch.full((B,), -1.0, device=device)

    for _ in range(p.get('max_depth', cfg_like.max_depth)):
        zz, aa = R.imagine(denoiser, zc, ac, p.get('horizon', cfg_like.horizon), B, K=6,
                           ctx_noise=p.get('ctx_noise', cfg_like.ctx_noise),
                           action_temp=p.get('action_temp', cfg_like.action_temp), generator=g)
        for h in range(zz.shape[1]): #shape zz: (B, H, N, D)
            peak = torch.maximum(peak, reward_T(zz[:, h:h + 1]).squeeze(-1))
        
        chunks.append(zz)
        zc = torch.cat([zc, zz], 1)[:, cfg_like.max_ctx:]
        ac = torch.cat([ac, aa], 1)[:, cfg_like.max_ctx:]
    
    return torch.cat(chunks, 1), peak.cpu().numpy()

In [ ]:
# --- random baseline: sample B rollouts, keep the trajectories, show the best one ---
rand_z, rp = random_best_rollout(epcfg, cz, ca)                 # (B, L, N, D), peaks (B,)
best = int(rp.argmax())                             # strongest random draw (of B)
rand_best_z = rand_z[best:best + 1]                 # (1, L, N, D)  -- swap for rand_z[0:1] for a single fixed draw
rand_roll_z = rand_z[0:1]                                   # (1, L, N, D)  -- swap for rand_z[best:best+1] for the best draw

plotComparison([('start ctx',              decode(cz)[0]),
                ('planned rollout',        decode(plan_z)[0]),
                ('random rollout',        decode(rand_roll_z)[0]),
                (f'greedy rollout \n(#{best} random rollout out of {len(rp)})', decode(rand_best_z)[0])],
               n_frames=8, title='MCTS plan VS random rollout VS greedy rollout')

plan_peak, _ = peak_along(plan_z[0])
plan_end_reward = reward_T(plan_z[:, -1:]).item()
random_best_end_reward = reward_T(rand_best_z[:, -1:]).item()
random_end_reward = reward_T(rand_roll_z[:, -1:]).item()
print(f'planned peak reward = {plan_peak:.3f}   random peak = {rp.mean():.3f} ± {rp.std():.3f}   '
      f'(start {s0:.3f})  -> {"PLANNING WINS" if plan_peak > rp.mean()+rp.std() else "~ random"}')
print(f'planned end reward = {plan_end_reward:.3f}   random best end reward = {random_best_end_reward:.3f}   '
      f'random end reward = {random_end_reward:.3f}')

## 2. Trying different configurations to improve planning

In [ ]:
CONFIGS = {
 'short/tight':  dict(horizon=16,  branching=4, max_depth=2, sim_horizon=16,  sim_rollouts=2, ctx_noise=0.3, action_temp=1.0),
 'short/wide':   dict(horizon=16,  branching=4, max_depth=2, sim_horizon=16,  sim_rollouts=2, ctx_noise=0.7, action_temp=1.0),
 'long/tight':   dict(horizon=28, branching=5, max_depth=3, sim_horizon=28,  sim_rollouts=3, ctx_noise=0.3, action_temp=1.0),
 'long/wide':    dict(horizon=28, branching=5, max_depth=3, sim_horizon=28,  sim_rollouts=3, ctx_noise=0.7, action_temp=1.0),
}

sweep = {}
print(f'start reward = {s0:.3f}\n')
for name, p in CONFIGS.items():
    ep = PlanConfig(K_steps=6, gamma=0.98, c_ucb=0.5, n_min=0, max_ctx=16,
                        n_iterations=20, **p)
    pl = MCTS(denoiser, reward_T, ep, seed=0); o = pl.plan(cz, ca)
    pz = torch.cat([e.z_seq for e in o['best_path']], 0)[None]
    plan_peak, _ = peak_along(pz[0])
    rp = random_best_rollout(ep, cz, ca, extra=p)[1]  # (B,) best reward per rollout
    # Part-I distinguishability of the root's edges under this config's edge knobs
    rm, _ = characterize_edges(cz, ca, H=p['horizon'], B=12,
                               ctx_noise=p['ctx_noise'], action_temp=p['action_temp'], seed=9)
    sweep[name] = dict(plan_peak=plan_peak, rand_mean=float(rp.mean()), rand_std=float(rp.std()),
                       gain=plan_peak - float(rp.mean()), root_rew_std=rm['rew_std'],
                       root_lat_vendi=rm['lat_vendi'], planner=pl, out=o, pz=pz)
    print(f"  {name:12s} plan_peak={plan_peak:.3f}  random={rp.mean():.3f}±{rp.std():.3f}  "
          f"gain={sweep[name]['gain']:+.3f}  root_rew_std={rm['rew_std']:.3f}")

names = list(sweep)
fig, ax = plt.subplots(figsize=(9, 4))
xx = np.arange(len(names))
ax.bar(xx - 0.2, [sweep[n]['plan_peak'] for n in names], 0.4, label='planned peak')
ax.bar(xx + 0.2, [sweep[n]['rand_mean'] for n in names], 0.4,
       yerr=[sweep[n]['rand_std'] for n in names], capsize=4, label='random peaks\n(averaged on 12 rollouts)')
ax.axhline(s0, color='k', ls='--', label=f'start reward ({s0:.2f})')
ax.set_xticks(xx); ax.set_xticklabels(names)
ax.set_ylabel('peak TCenterReward')
ax.legend()
ax.set_title('does planning beat random? (per config)')
plt.tight_layout()
plt.show()

In [ ]:
best_name = "long/wide"
pz = sweep[best_name]['pz']
full = torch.cat([cz, pz], 1)
idxs = np.linspace(0, full.shape[1] - 1, 8, dtype=int)
strip = []
for t in idxs:
    rgb = frame_rgb(full[:, t:t + 1]); _, dbg = score_t_centered(rgb, **SCORE_KW)
    strip.append(annotate_t(rgb, dbg))
plt.figure(figsize=(16, 2.6))
plt.imshow(np.concatenate(strip, 1)); plt.axis('off')
plt.title(f'plan by config "{best_name}"')
plt.tight_layout()
plt.show()

## 3. Visualization of plans (MCTS plan / greedy rollout / random rollout)

In [ ]:
SCORE_KW = dict(center_xy=(0.5, 0.5), sigma=0.25)
reward_T = TCenterReward(decode_fn=decode, **SCORE_KW)

WINDOW_SEED=1234    # t0=0, WINDOW_SEED= 3000
t0 = 50
imgs, actions, latents = get_window(WINDOW_SEED)
plotVideo(imgs[0], fps=10)

# cz shapes: (1, T_ctx, N, D), ca shapes: (1, T_ctx, n_act)
wcz, wca, wgt = make_state(t0=t0, Tc=T_CTX, window_seed=WINDOW_SEED, visualize=True)


s0, d0 = score_t_centered(frame_rgb(wcz[:, -1:]), **SCORE_KW)
plt.figure(figsize=(4, 4))
plt.imshow(annotate_t(frame_rgb(wcz[:, -1:]), d0))
plt.axis('off')
plt.title(f'last context frame\nR={s0:.2f}  center={d0["center"]:.2f}')
plt.tight_layout()
plt.show()
print(f'start reward = {s0:.3f}')

In [ ]:
# long/wide config 
work_cfg = PlanConfig(horizon=28, branching=5, max_depth=3, sim_horizon=28, sim_rollouts=3,
                          ctx_noise=0.7, action_temp=1.0, n_iterations=28, K_steps=6,
                          gamma=0.98, c_ucb=0.5, n_min=0, max_ctx=16)

t = time.time()
wout = MCTS(denoiser, reward_T, work_cfg, seed=0).plan(wcz, wca, verbose=True)
work_pz = torch.cat([e.z_seq for e in wout['best_path']], 0)[None]
wfull = torch.cat([wcz, work_pz], 1)

wrgb, wrew = [], []   # wrgb = annotated frames, wrew = TCenterReward scores
for tt in range(wfull.shape[1]):
    rgb = frame_rgb(wfull[:, tt:tt + 1]) 
    s, d = score_t_centered(rgb, **SCORE_KW)
    wrgb.append(annotate_t(rgb, d))
    wrew.append(s)

wrew = np.array(wrew)
Tcw = wcz.shape[1]
peak_t = int(Tcw + np.argmax(wrew[Tcw:]))
print(f'planned in {time.time()-t:.0f}s | start R={wrew[Tcw-1]:.3f} -> plan peak R={wrew[peak_t]:.3f} '
      f'(gain {wrew[peak_t]-wrew[Tcw-1]:+.3f})  plan length={work_pz.shape[1]} frames')

fig = plt.figure(figsize=(15, 6.4))
gs = fig.add_gridspec(2, 8, height_ratios=[1.25, 1.0], hspace=0.3, wspace=0.12)
samp = np.unique(np.concatenate([[Tcw - 1], np.linspace(Tcw, wfull.shape[1] - 1, 7, dtype=int)]))[:8]

for k, tt in enumerate(samp):
    ax = fig.add_subplot(gs[0, k])
    ax.imshow(wrgb[tt])
    ax.axis('off')
    tag = 'start' if tt == Tcw - 1 else ('PEAK' if tt == peak_t else f't={tt-Tcw+1}')
    ax.set_title(f'{tag}\nR={wrew[tt]:.2f}', fontsize=9,
                 color=('C2' if tt == peak_t else ('C3' if tt == Tcw - 1 else 'k')))

axc = fig.add_subplot(gs[1, 0:5])
axc.plot(range(wfull.shape[1]), wrew, 'o-', ms=4)
axc.axvspan(0, Tcw - 1, color='grey', alpha=0.15, label='context')
axc.axvline(Tcw - 0.5, color='grey', ls='--')
axc.scatter([Tcw - 1], [wrew[Tcw - 1]], c='C3', s=90, zorder=5, label=f'start R={wrew[Tcw-1]:.2f}')
axc.scatter([peak_t], [wrew[peak_t]], c='C2', s=120, marker='*', zorder=5, label=f'plan peak R={wrew[peak_t]:.2f}')
axc.set_xlabel('frame (context | planned rollout)'); axc.set_ylabel('TCenterReward')
axc.legend(fontsize=9, loc='lower right'); axc.grid(alpha=.3)
axc.set_title(f'MCTS plan raises the reward (gain {wrew[peak_t]-wrew[Tcw-1]:+.2f})')
axb = fig.add_subplot(gs[1, 5]); axb.imshow(wrgb[Tcw - 1]); axb.axis('off')
axb.set_title(f'START\nR={wrew[Tcw-1]:.2f}', color='C3')
axp = fig.add_subplot(gs[1, 6:8]); axp.imshow(wrgb[peak_t]); axp.axis('off')
axp.set_title(f'PLANNED (peak)\nR={wrew[peak_t]:.2f}', color='C2')
fig.suptitle('MCTS plan in the imagination of the WorldModel\nconfig used: (horizon=28, branching=5, max_depth=3, sim_horizon=28,  sim_rollouts=3, ctx_noise=0.7)', y=0.99)
plt.show()

# inline video of the planned rollout (holds a beat on the peak frame)
mediapy.show_video([np.ascontiguousarray(r) for r in wrgb] + [wrgb[peak_t]] * 6, fps=6)

In [ ]:
# ---- reusable: the plan-cell visualization, for ANY rollout -----------------
def show_rollout(ctx_z, roll_z, name='rollout', headline=None):
    """Filmstrip + reward curve + start/peak insets + video for  ctx + a rollout.
    ctx_z (1,Tc,N,D), roll_z (1,L,N,D). Returns the per-frame reward array."""
    full = torch.cat([ctx_z, roll_z], 1)
    rgb_l, rew = [], []
    for tt in range(full.shape[1]):
        rgb = frame_rgb(full[:, tt:tt + 1]); s, d = score_t_centered(rgb, **SCORE_KW)
        rgb_l.append(annotate_t(rgb, d)); rew.append(s)
    rew = np.array(rew); Tc = ctx_z.shape[1]; peak_t = int(Tc + np.argmax(rew[Tc:]))
    print(f'{name}: start R={rew[Tc-1]:.3f} -> peak R={rew[peak_t]:.3f} '
          f'(gain {rew[peak_t]-rew[Tc-1]:+.3f})  length={roll_z.shape[1]} frames')

    fig = plt.figure(figsize=(15, 6.4))
    gs = fig.add_gridspec(2, 8, height_ratios=[1.25, 1.0], hspace=0.3, wspace=0.12)
    samp = np.unique(np.concatenate([[Tc - 1], np.linspace(Tc, full.shape[1] - 1, 7, dtype=int)]))[:8]
    for k, tt in enumerate(samp):
        ax = fig.add_subplot(gs[0, k]); ax.imshow(rgb_l[tt]); ax.axis('off')
        tag = 'start' if tt == Tc - 1 else ('PEAK' if tt == peak_t else f't={tt-Tc+1}')
        ax.set_title(f'{tag}\nR={rew[tt]:.2f}', fontsize=9,
                     color=('C2' if tt == peak_t else ('C3' if tt == Tc - 1 else 'k')))
    axc = fig.add_subplot(gs[1, 0:5])
    axc.plot(range(full.shape[1]), rew, 'o-', ms=4)
    axc.axvspan(0, Tc - 1, color='grey', alpha=0.15, label='context')
    axc.axvline(Tc - 0.5, color='grey', ls='--')
    axc.scatter([Tc - 1], [rew[Tc - 1]], c='C3', s=90, zorder=5, label=f'start R={rew[Tc-1]:.2f}')
    axc.scatter([peak_t], [rew[peak_t]], c='C2', s=120, marker='*', zorder=5, label=f'{name} peak R={rew[peak_t]:.2f}')
    axc.set_xlabel(f'frame (context | {name})'); axc.set_ylabel('TCenterReward')
    axc.legend(fontsize=9, loc='lower right'); axc.grid(alpha=.3)
    axc.set_title(f'{name}: reward vs frame (gain {rew[peak_t]-rew[Tc-1]:+.2f})')
    axb = fig.add_subplot(gs[1, 5]); axb.imshow(rgb_l[Tc - 1]); axb.axis('off')
    axb.set_title(f'START\nR={rew[Tc-1]:.2f}', color='C3')
    axp = fig.add_subplot(gs[1, 6:8]); axp.imshow(rgb_l[peak_t]); axp.axis('off')
    axp.set_title(f'{name.upper()} (peak)\nR={rew[peak_t]:.2f}', color='C2')
    fig.suptitle(headline or f'{name} + TCenterReward (green=image center, yellow=T)', y=0.99)
    plt.show()
    mediapy.show_video([np.ascontiguousarray(r) for r in rgb_l] + [rgb_l[peak_t]] * 6, fps=6)
    return rew


rand_z, rp = random_best_rollout(work_cfg, wcz, wca, B=12, seed=0)
best = int(rp.argmax())

# (1) the random BEST rollout — strongest of the B draws (argmax peak)
show_rollout(wcz, rand_z[best:best + 1], name='best random',
             headline=f'Greedy rollout\nBest random rollout (of {len(rp)}) - no planning')

# (2) a single random rollout — one unselected draw
show_rollout(wcz, rand_z[0:1], name='greedy rollout',
             headline='A single random rollout - no planning')


# Part III: Tree metrics

The tree statistics that actually diagnose exploration. The most on-target ones:

- **visit-count entropy** over the root's children (normalised to `[0,1]`): `1` =
  visits spread uniformly = the search could not tell the children apart; low =
  it committed. This is the *tree-level shadow* of edge collapse.
- **root-child value spread** (`max − mean`, and std): the decision resolution on
  the first action.
- **realized nodes per depth / mean visited depth / best-path depth**.
- **visit↔value correlation** across siblings: is UCB spending visits where the
  value is? (`nan` if too few visited children.)
- **edge-value (terminal reward) std inside the tree** vs a random batch: does the
  search preferentially expand distinguishable regions?

In [ ]:
def tree_metrics(planner, out=None):
    root = planner.root
    ch = root.children
    visits = np.array([c.n_visit for c in ch], float)
    vals   = np.array([c.value if c.n_visit > 0 else np.nan for c in ch], float)
    tot = max(visits.sum(), 1.0); p = visits / tot; nz = p[p > 0]
    visit_entropy = float(-(nz * np.log(nz)).sum() / np.log(len(ch))) if len(ch) > 1 else 0.0
    fv = vals[np.isfinite(vals)]
    val_spread = float(fv.max() - fv.mean()) if fv.size else 0.0
    depths = [n.depth for n in planner.all_nodes]; maxd = max(depths)
    per_depth = {d: sum(x == d for x in depths) for d in range(maxd + 1)}
    visited = [n for n in planner.all_nodes if n.n_visit > 0]
    vc = [(c.n_visit, c.value) for c in ch if c.n_visit > 0]
    _vv = np.array([x[0] for x in vc], float); _qq = np.array([x[1] for x in vc], float)
    corr = (float(np.corrcoef(_vv, _qq)[0, 1])                 # is UCB spending visits where value is?
            if len(vc) > 2 and _vv.std() > 0 and _qq.std() > 0 else float('nan'))
    best_first = out['best_path'][0] if (out and out['best_path']) else max(ch, key=lambda c: c.value)
    ev = np.array([n.edge_val for n in planner.all_nodes if n.parent is not None])
    return dict(n_nodes=len(planner.all_nodes), n_root_children=len(ch),
                visit_entropy=visit_entropy, val_spread=val_spread, val_std=float(np.nanstd(vals)),
                per_depth=per_depth, mean_depth_visited=float(np.mean([n.depth for n in visited])) if visited else 0.0,
                best_depth=len(out['best_path']) if out else None,
                commit=float(best_first.n_visit / tot), visit_value_corr=corr,
                edge_val_std=float(ev.std()), edge_val_range=float(ev.max() - ev.min()),
                visits=visits, vals=vals)

def display_tree_metrics(tm0):
    for k, v in tm0.items():
        if k not in ('visits', 'vals'):
            print(f'  {k:20s} {v}')

tm0 = tree_metrics(planner0, out0)
display_tree_metrics(tm0)

## 1. Contrast: a *branchable* tree vs a *collapsed* tree

The collapsed tree should show **high visit entropy, ~zero value spread**: the planner is flying blind.

In [ ]:
#long/wide config (from the sweep above)
CFG_BRANCH = dict(horizon=28, branching=5, max_depth=3, sim_horizon=28, sim_rollouts=3, 
                  ctx_noise=0.7, action_temp=1.0)
CFG_COLLAPSE = dict(horizon=6, branching=5, max_depth=3, sim_horizon=6, sim_rollouts=3,
                    ctx_noise=0.0, action_temp=1.0)
trees = {}
for name, p in [('branchable', CFG_BRANCH), ('collapsed', CFG_COLLAPSE)]:
    ep = PlanConfig(K_steps=6, gamma=0.98, c_ucb=0.5, n_min=0, max_ctx=24,
                        n_iterations=24, **p)
    pl = MCTS(denoiser, reward_T, ep, seed=0)
    o = pl.plan(cz, ca)
    trees[name] = dict(m=tree_metrics(pl, o), pl=pl, o=o)
    m = trees[name]['m']
    print(f"{name:11s} visit_entropy={m['visit_entropy']:.2f}  val_spread={m['val_spread']:.3f}  "
          f"edge_val_std={m['edge_val_std']:.3f}  commit={m['commit']:.2f}  "
          f"visit~value corr={m['visit_value_corr']:.2f}")
    display_tree_metrics(m)
    
    # Display plan and plot video
    pz = torch.cat([e.z_seq for e in o['best_path']], 0)[None]
    show_rollout(cz, pz, name=f'plan ({name})', headline=f'Plan rollout ({name}) — {len(o["best_path"])} edges, {pz.shape[1]} frames')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
# (a) root-child visits & values, branchable
for col, name in enumerate(['branchable', 'collapsed']):
    m = trees[name]['m']; ch = trees[name]['pl'].root.children
    order = np.argsort(-m['visits'])
    a = ax[0, col]
    a.bar(range(len(ch)), m['visits'][order], color='C0', alpha=0.6, label='visits')
    a.set_ylabel('root-child visits', color='C0'); a.set_xlabel('root child (sorted)')
    a.set_title(f"{name}:  visit_entropy={m['visit_entropy']:.2f}, val_spread={m['val_spread']:.3f}")
    a2 = a.twinx()
    a2.plot(range(len(ch)), m['vals'][order], 'o-', color='C3', label='value')
    a2.set_ylabel('root-child value', color='C3')
# (b) edge-value spread inside tree vs random batch
gb = torch.Generator(device=device).manual_seed(11)
zz, _ = R.imagine(denoiser, cz, ca, 8, 32, K=6, ctx_noise=0.6, action_temp=2.5, generator=gb)
rand_terminal = reward_T(zz[:, -1]).cpu().numpy()
for col, name in enumerate(['branchable', 'collapsed']):
    ev = np.array([n.edge_val for n in trees[name]['pl'].all_nodes if n.parent is not None])
    a = ax[1, col]
    a.hist(ev, bins=15, alpha=0.6, label='tree edge terminal reward', density=True)
    a.hist(rand_terminal, bins=15, alpha=0.4, label='random batch', density=True)
    a.set_xlabel('terminal reward'); a.set_ylabel('density'); a.legend(fontsize=8)
    a.set_title(f'{name}: edge_val_std={trees[name]["m"]["edge_val_std"]:.3f}')
plt.tight_layout(); plt.show()